# Multi-seed protocol — federated hybrid MLP/LightGBM on CIC-IoT-2023

Runs the full experiment once per seed (42-46) and writes confidence intervals
and paired significance tests to `multi_seed_summary.json`.

Attach two datasets before running:

* `fedlgbm-src` — the source package (`scripts/push_to_kaggle.py`)
* `ciciot2023-fedlgbm-parts` — the exact CSV parts (`scripts/push_data_to_kaggle.py`)

Run it with **Save Version -> Save & Run All (Commit)**, not interactively — an
interactive session idles out long before five seeds finish.

In [ ]:
import sys, time, json, shutil
from pathlib import Path

INPUT = Path('/kaggle/input')
OUT = Path('/kaggle/working/outputs')

print('attached:', [p.name for p in INPUT.iterdir()] if INPUT.exists() else 'NOTHING')

# Locate the package rather than assuming a mount layout: Kaggle's extraction
# of an uploaded zip does not always preserve the directory level the local
# tree had, and a wrong guess here fails two cells later as ModuleNotFoundError.
hits = sorted(INPUT.glob('*/**/fedlgbm/__init__.py'))
assert hits, ('fedlgbm not found under /kaggle/input — attach the fedlgbm-src '
              'dataset via Add Data -> Your Datasets')
SRC = hits[0].parent.parent
sys.path.insert(0, str(SRC))
print('source root:', SRC)

# Same for the data: match on content, not on the dataset's folder name.
csv_dirs = {p.parent for p in INPUT.glob('*/**/*.csv')}
csv_dirs = {d for d in csv_dirs if len(list(d.glob('*.csv'))) >= 5}
assert csv_dirs, ('no CSV parts found under /kaggle/input — attach the '
                  'ciciot2023-fedlgbm-parts dataset')
DATA = sorted(csv_dirs)[0]
parts = sorted(DATA.glob('*.csv'))
print(f'data: {DATA}  ({len(parts)} parts, '
      f'{sum(f.stat().st_size for f in parts)/1024/1024:,.1f} MB)')

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
import torch

# Preflight. A P100 (sm_60) with a cu128 torch build reports cuda.is_available()
# as True and then fails every kernel launch with "no kernel image is available
# for execution on the device". Without this check that surfaces as a per-model
# error hours later, after the sweep has already run to completion around it.
DEVICE = 'cpu'
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    arch_list = torch.cuda.get_arch_list()
    print(f'gpu: {torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}')
    print(f'torch {torch.__version__} built for: {arch_list}')
    try:
        (torch.zeros(8, 8, device='cuda') @ torch.ones(8, 8, device='cuda')).sum().item()
        DEVICE = 'cuda'
        print('CUDA preflight passed')
    except Exception as exc:
        raise RuntimeError(
            f'CUDA is present but unusable: {exc}\n\n'
            f'This torch build has no kernels for sm_{cap[0]}{cap[1]}. Switch the '
            'accelerator (Settings -> Accelerator) to "GPU T4 x2", which is sm_75 '
            'and supported. Do not run on CPU: a full seed takes roughly four '
            'times longer, and mixing devices across seeds invalidates the '
            'aggregate.'
        ) from exc
else:
    raise RuntimeError('No GPU attached. Settings -> Accelerator -> GPU T4 x2.')

In [ ]:
from fedlgbm import ExperimentConfig, setup_logging
from fedlgbm.ablations import run_multi_seed

setup_logging()

# Mirrors the local command line exactly. Any divergence here makes the
# intervals incomparable with the single-seed numbers already in the paper.
cfg = ExperimentConfig()
cfg.name = 'multiseed_gpu'
cfg.output_dir = OUT
cfg.device = DEVICE
cfg.skip_privacy_attacks = True

cfg.data.data_dir = DATA
cfg.data.label_mode = 'category'
cfg.data.max_rows = None
cfg.data.max_files = None
cfg.data.max_per_class = 50_000

cfg.partition.n_clients = 15
cfg.fed.n_rounds = 30
cfg.fed.local_epochs = 2

cfg.mlp.hidden_dims = (256, 128)
cfg.mlp.embedding_dim = 64

cfg.lgbm.class_weight = None
cfg.lgbm.num_leaves = 255
cfg.lgbm.n_estimators = 1000

SEEDS = (42, 43, 44, 45, 46)
SKIP = {'fed_hybrid_mlp_lgbm', 'centralized_mlp_lgbm', 'fedavg_mlp_only',
        'logistic_regression', 'lstm', 'isolation_forest'}

print(json.dumps(cfg.to_dict(), indent=2)[:1200])

In [ ]:
t0 = time.time()
# REFERENCE must be a model that actually runs. The library default is
# fed_hybrid_mlp_lgbm, which SKIP excludes here, and pairing against an absent
# model yields a summary with no significance test at all.
summary = run_multi_seed(cfg, seeds=SEEDS, skip=SKIP,
                         reference='stacked_mlp_lgbm')
print(f'\ntotal {(time.time()-t0)/3600:.2f} h')

In [ ]:
# Surface the two things the manuscript needs, then flatten the run directory
# into /kaggle/working so both survive as notebook output.
agg = summary['aggregated']
print(f"n_seeds = {agg['n_seeds']}")
for model, metrics in sorted(agg['models'].items()):
    acc, f1 = metrics['accuracy'], metrics['f1_macro']
    print(f"{model:24s} acc {acc['mean']:.4f} "
          f"[{acc['ci95_low']:.4f}, {acc['ci95_high']:.4f}]   "
          f"macro-F1 {f1['mean']:.4f} "
          f"[{f1['ci95_low']:.4f}, {f1['ci95_high']:.4f}]")

print('\n--- accuracy, paired ---')
print(json.dumps(summary['significance_accuracy'], indent=2))
print('\n--- macro-F1, paired ---')
print(json.dumps(summary['significance_f1_macro'], indent=2))

run_dir = OUT / cfg.name
for name in ('multi_seed_summary.json', 'multi_seed_table.md'):
    src = run_dir / name
    if src.exists():
        shutil.copy2(src, Path('/kaggle/working') / name)
        print('kept', name)

shutil.make_archive('/kaggle/working/multiseed_outputs', 'zip', OUT)
print('archived outputs')